# EMG Signal Analysis

**Pipeline** (applied to every channel in the selected folder):
1. Raw signal
2. → Bandpass filter (6–500 Hz, 4th-order Butterworth, zero-phase `filtfilt`)
3. → Feedforward comb filter: `y[n] = x[n] − x[n−M]`, `M = round(fs/50)` — notches at 50, 100, 150 … Hz

**Output — 3 interactive Plotly figures:**
- **Fig 1 — Time domain:** rows = channels, cols = Raw | Bandpass | Comb
- **Fig 2 — FFT:** same grid, frequency domain
- **Fig 3 — FFT overlay:** all 3 stages overlaid per channel with 50 Hz harmonic markers

Run cells top to bottom, then use the widget in the last cell.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as sig

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, clear_output

RECORDINGS_DIR = Path("recordings")

In [ ]:
# ── data ─────────────────────────────────────────────────────────────────────

def load_signal(folder: str, channel: str):
    path = RECORDINGS_DIR / folder / f"{channel}.csv"
    df = pd.read_csv(path)
    t = df["time"].values
    x = df["data"].values
    fs = round(1.0 / (t[1] - t[0]))
    return t, x, float(fs)

# ── filters ───────────────────────────────────────────────────────────────────

def bandpass_filter(x: np.ndarray, fs: float,
                    low_hz: float = 6.0, high_hz: float = 500.0,
                    order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth bandpass — SOS form for numerical stability."""
    sos = sig.butter(order, [low_hz, high_hz], btype="band", fs=fs, output="sos")
    return sig.sosfiltfilt(sos, x)


def fir_notch_filter(x: np.ndarray, fs: float,
                     f0: float = 50.0, bw: float = 5.0,
                     max_hz: float = 500.0) -> np.ndarray:
    """Cascade of FIR bandstop filters at each harmonic of f0.
    Design: firwin + Hamming window.

    Uses fftconvolve + delay compensation instead of filtfilt.
    filtfilt internally allocates an (n_taps × n_taps) matrix for lfilter_zi,
    which causes a MemoryError at audio sample rates (37 GB at 44 kHz, bw=5 Hz).
    For a symmetric (linear-phase) FIR the group delay is constant at
    (n_taps-1)//2 samples, so trimming the fftconvolve output by that offset
    gives the same result without the memory cost.
    """
    n_taps = int(8 * fs / bw)
    if n_taps % 2 == 0:
        n_taps += 1
    delay = (n_taps - 1) // 2
    y = x.copy().astype(float)
    k = 1
    while k * f0 <= max_hz and k * f0 < fs / 2:
        fc = k * f0
        b = sig.firwin(n_taps,
                       [max(fc - bw / 2, 0.5), min(fc + bw / 2, fs / 2 - 0.5)],
                       window="hamming", pass_zero=True, fs=fs)
        y_full = sig.fftconvolve(y, b, mode="full")   # O(N log N), no zi matrix
        y = y_full[delay : delay + len(x)]
        k += 1
    return y


def iir_notch_filter(x: np.ndarray, fs: float,
                     f0: float = 50.0, Q: float = 30.0,
                     max_hz: float = 500.0) -> np.ndarray:
    """Cascade of 2nd-order IIR biquad notch filters at harmonics of f0.
    Applied zero-phase (filtfilt). Sharpest notch, flattest passband."""
    y = x.copy().astype(float)
    k = 1
    while k * f0 <= max_hz and k * f0 < fs / 2:
        b, a = sig.iirnotch(k * f0, Q, fs=fs)
        y = sig.filtfilt(b, a, y)
        k += 1
    return y


def fir_comb_filter(x: np.ndarray, fs: float, f0: float = 50.0) -> np.ndarray:
    """Feedforward FIR comb: y[n] = x[n] − x[n−M],  M = round(fs/f0).
    Nulls at all harmonics of f0 in one pass; applied zero-phase (filtfilt).
    Wide notches — passband follows |sin(π·f·M/fs)|, not flat."""
    M = int(round(fs / f0))
    b = np.zeros(M + 1)
    b[0] = 1.0
    b[M] = -1.0
    return sig.filtfilt(b, [1.0], x.astype(float))


def iir_comb_filter(x: np.ndarray, fs: float,
                    f0: float = 50.0, r: float = 0.98) -> np.ndarray:
    """Feedback IIR comb: y[n] = x[n] − x[n−M] + r^M·y[n−M].
    Poles near zeros → sharp narrow notches, flat passband.
    Applied zero-phase (filtfilt). Requires r < 1 for stability."""
    M = int(round(fs / f0))
    rM = r ** M
    b = np.zeros(M + 1)
    a = np.zeros(M + 1)
    b[0] = 1.0;  b[M] = -1.0   # numerator:   1 − z^{−M}
    a[0] = 1.0;  a[M] = -rM    # denominator: 1 − r^M·z^{−M}
    return sig.filtfilt(b, a, x.astype(float))

# ── FFT ───────────────────────────────────────────────────────────────────────

def rfft_single_sided(x: np.ndarray, fs: float):
    N = len(x)
    freqs = np.fft.rfftfreq(N, d=1.0 / fs)
    mag = np.abs(np.fft.rfft(x)) * (2.0 / N)
    mag[0] /= 2.0
    if N % 2 == 0:
        mag[-1] /= 2.0
    return freqs, mag

# ── palette ───────────────────────────────────────────────────────────────────

STAGE_STYLES = [
    # (label, line_color, fill_color, time_key, fft_key)
    ("Raw",               "#1565C0", "rgba(21,101,192,0.10)",  "raw", "fft_raw"),
    ("Bandpass 6–500 Hz", "#E65100", "rgba(230,81,0,0.10)",    "bp",  "fft_bp"),
    ("Filtered",          "#2E7D32", "rgba(46,125,50,0.10)",   "flt", "fft_flt"),
]

CHANNELS = [
    ("emg_line_L", "Line In — Left"),
    ("emg_line_R", "Line In — Right"),
    ("emg_mic",    "Microphone"),
]

FILTER_LABELS = {
    "fir_notch": "Bandpass + FIR Notch  (firwin, bw=5 Hz, linear-phase)",
    "iir_notch": "Bandpass + IIR Notch  (iirnotch, Q=30, zero-phase)",
    "fir_comb":  "Bandpass + FIR Comb   y[n]=x[n]−x[n−M]  (zero-phase)",
    "iir_comb":  "Bandpass + IIR Comb   +rᴹ·y[n−M], r=0.98  (zero-phase)",
}

# ── main ──────────────────────────────────────────────────────────────────────

def analyze_folder(folder: str, fft_max_hz: float = 1000.0,
                   filter_type: str = "iir_notch"):
    """Load every channel in *folder*, apply bandpass + selected filter, and plot."""

    filter_label = FILTER_LABELS[filter_type]
    stage_styles = [
        STAGE_STYLES[0],
        STAGE_STYLES[1],
        (filter_label, *STAGE_STYLES[2][1:]),
    ]

    # ── load & filter all channels ────────────────────────────────────────────
    chs = []
    for ch_key, ch_label in CHANNELS:
        path = RECORDINGS_DIR / folder / f"{ch_key}.csv"
        if not path.exists():
            print(f"  Skipping {ch_label}: {ch_key}.csv not found")
            continue
        print(f"Loading {ch_key}.csv …", end="  ")
        t, raw, fs = load_signal(folder, ch_key)
        print(f"fs={fs:.0f} Hz  {len(raw):,} samples  {t[-1]:.3f} s")

        bp = bandpass_filter(raw, fs)

        print(f"  {filter_label} …")
        if filter_type == "fir_notch":
            flt = fir_notch_filter(bp, fs)
        elif filter_type == "iir_notch":
            flt = iir_notch_filter(bp, fs)
        elif filter_type == "fir_comb":
            flt = fir_comb_filter(bp, fs)
        else:  # iir_comb
            flt = iir_comb_filter(bp, fs)

        fr, fft_raw = rfft_single_sided(raw, fs)
        _,  fft_bp  = rfft_single_sided(bp,  fs)
        _,  fft_flt = rfft_single_sided(flt, fs)

        chs.append(dict(
            label=ch_label,
            time=t, raw=raw, bp=bp, flt=flt,
            fr=fr, fft_raw=fft_raw, fft_bp=fft_bp, fft_flt=fft_flt,
        ))

    if not chs:
        print("No channel files found in", folder)
        return

    n_harm = min(int(fft_max_hz / 50), 20)

    for cd in chs:
        fr = cd["fr"]
        fm = fr <= fft_max_hz

        # ── Figure A: 3 rows (stages) × 2 cols (time | FFT) ──────────────────
        fig_a = make_subplots(
            rows=3, cols=2,
            subplot_titles=[
                "Raw — time",               "Raw — FFT",
                "Bandpass 6–500 Hz — time", "Bandpass 6–500 Hz — FFT",
                f"{filter_label} — time",   f"{filter_label} — FFT",
            ],
            column_widths=[0.55, 0.45],
            vertical_spacing=0.10,
            horizontal_spacing=0.08,
        )

        for r, (sl, color, fill, sig_key, fft_key) in enumerate(stage_styles, 1):
            fig_a.add_trace(go.Scatter(
                x=cd["time"], y=cd[sig_key],
                name=sl, line=dict(width=0.6, color=color),
                showlegend=False,
            ), row=r, col=1)
            fig_a.add_trace(go.Scatter(
                x=fr[fm], y=cd[fft_key][fm],
                name=sl, line=dict(width=1.0, color=color),
                fill="tozeroy", fillcolor=fill,
                showlegend=False,
            ), row=r, col=2)
            fig_a.update_xaxes(title_text="Time (s)",       row=r, col=1)
            fig_a.update_xaxes(title_text="Frequency (Hz)", row=r, col=2)
            fig_a.update_yaxes(title_text="Amplitude (V)",  row=r, col=1)
            fig_a.update_yaxes(title_text="Magnitude (V)",  row=r, col=2)

        fig_a.update_layout(
            title_text=f"{cd['label']} — {folder}  [{filter_label}]",
            height=700,
            template="plotly_white",
        )
        fig_a.show()

        # ── Figure B: FFT overlay + harmonic markers ───────────────────────────
        fig_b = go.Figure()
        for sl, color, _, _, fft_key in stage_styles:
            fig_b.add_trace(go.Scatter(
                x=fr[fm], y=cd[fft_key][fm],
                name=sl, line=dict(width=1.2, color=color),
            ))

        for k in range(1, n_harm + 1):
            fig_b.add_vline(
                x=k * 50,
                line=dict(color="rgba(180,0,0,0.25)", width=1, dash="dot"),
                annotation_text=str(k * 50),
                annotation_position="top right",
                annotation_font_size=8,
                annotation_font_color="rgba(180,0,0,0.55)",
            )

        fig_b.update_layout(
            title_text=f"{cd['label']} — FFT Overlay — {folder}  [{filter_label}]",
            xaxis_title="Frequency (Hz)",
            yaxis_title="Magnitude (V)",
            height=380,
            template="plotly_white",
            legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
        )
        fig_b.show()

    print("Done.")

In [ ]:
clear_output(wait=True)

folders = sorted([f.name for f in RECORDINGS_DIR.iterdir() if f.is_dir()])

if not folders:
    print("No recording folders found in 'recordings/'. Run dual_acquisition.py first.")
else:
    try:
        _emg_run_btn
    except NameError:
        _emg_folder_w = widgets.Dropdown(
            options=folders,
            value=folders[-1],
            description="Recording:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="440px"),
        )
        _emg_filter_w = widgets.ToggleButtons(
            options=[
                ("FIR Notch",     "fir_notch"),
                ("IIR Notch",     "iir_notch"),
                ("FIR Comb (FF)", "fir_comb"),
                ("IIR Comb (FB)", "iir_comb"),
            ],
            value="iir_notch",
            description="Filter:",
            style={"description_width": "initial", "button_width": "130px"},
            tooltips=[
                "FIR bandstop cascade — linear phase, many taps, slow at 44 kHz",
                "IIR biquad notch cascade — sharpest notch, flat passband",
                "Feedforward comb y[n]=x[n]−x[n−M] — wide notches, non-flat passband",
                "Feedback IIR comb +rᴹ·y[n−M] — sharp notches, flat passband",
            ],
        )
        _emg_fft_w = widgets.BoundedIntText(
            value=1000, min=100, max=24000, step=50,
            description="FFT max Hz:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="220px"),
        )
        _emg_run_btn = widgets.Button(
            description="▶  Analyze & Plot",
            button_style="primary",
            layout=widgets.Layout(width="190px", height="38px"),
        )
        _emg_out = widgets.Output()
    else:
        _emg_folder_w.options = folders
        _emg_folder_w.value = folders[-1]

    def _on_run(b):
        with _emg_out:
            clear_output(wait=True)
            if _emg_filter_w.value == "fir_notch":
                print("⚠ FIR Notch: ~8·fs/bw taps per harmonic — may take 30–60 s at 44 kHz.")
            analyze_folder(
                _emg_folder_w.value,
                fft_max_hz=float(_emg_fft_w.value),
                filter_type=_emg_filter_w.value,
            )

    _emg_run_btn._click_handlers.callbacks[:] = []
    _emg_run_btn.on_click(_on_run)

    display(widgets.VBox([
        widgets.HTML(
            "<b>Select a recording and filter, then click Analyze &amp; Plot.<br>"
            "All channels (Line L, Line R, Mic) are plotted automatically.</b>"
        ),
        _emg_folder_w,
        _emg_filter_w,
        _emg_fft_w,
        _emg_run_btn,
        _emg_out,
    ]))